# Kandor OpenAI Backend Quickstart

This notebook runs the same world-building pipeline as the mock quickstart, but with a real OpenAI-compatible backend.

It expects credentials in the current environment or in the repository `.env` file. Supported variables: `OPENAI_API_KEY`, optional `OPENAI_MODEL`, optional `OPENAI_BASE_URL`.


In [1]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / 'kandor').exists():
        sys.path.insert(0, str(candidate))
        break


In [2]:
from kandor import LoreGenerator, SimulationRunner, WorldBuilder
from kandor.core.models import Event
from kandor.llm.openai import OpenAICompatibleLLM


In [3]:
llm = OpenAICompatibleLLM()
{
    'model': llm.config.model,
    'base_url': llm.config.base_url,
    'app_name': llm.config.app_name,
}


{'model': 'gpt-4.1-mini',
 'base_url': 'https://api.openai.com/v1',
 'app_name': 'sim-llm-game'}

In [4]:
builder = WorldBuilder(llm=llm)
blueprint = builder.create_world(
    prompt="Create a compact fantasy world with 2 factions, 2 locations, one tension, and one seed event."
)
memory = builder.create_memory(blueprint)
runner = SimulationRunner(world=blueprint.spec, memory=memory)


In [5]:
blueprint


WorldBlueprint(spec=WorldSpec(premise='In a compact fantasy world, the kingdom of Eldoria and the enigmatic clan of Shadowmere vie for control over two key locations: the mystic Glimmerwood Forest and the ancient Ironhold Fortress. Tensions rise as both factions aim to uncover the secrets hidden within these lands, leading to an uneasy standoff that threatens to erupt into open conflict.', genre='Fantasy', rules=['Magic exists in limited forms, accessible primarily through artifacts and ancient sites.', 'Factions compete for control over locations to gain strategic advantages.', 'Events have lasting impacts on faction relations and territorial control.'], ontology={'Entities': ['character', 'faction', 'location', 'artifact', 'event'], 'Relations': ['controls', 'occupies', 'allies_with', 'enemies_with', 'influences'], 'Events': ['conflict', 'discovery', 'alliance', 'betrayal']}, simulation_parameters={'turn_length': '1 month', 'starting_year': 1001}), entities=[Entity(id='faction_eldori

In [6]:
participants = [entity.id for entity in blueprint.entities[:2]]

candidate = Event(
    id="event.openai.step1",
    type="alliance_formed",
    timestamp=1,
    participants=participants,
    effects=[],
    summary="A tentative alliance forms during the OpenAI notebook walkthrough.",
)

runner.step([candidate])
runner.state


WorldState(current_time=1, entity_ids=['faction_eldoria', 'faction_shadowmere'], derived_state={})

In [7]:
lore = LoreGenerator(llm=llm)
lore.summarize_world(memory=runner.memory, time=runner.state.current_time)


'At time 1, there are no active relations among the entities. However, a tentative alliance was formed during the OpenAI notebook walkthrough, suggesting initial cooperation despite the lack of ongoing active relations.'